# Notebook 1: Building the Telemetry Layer

Part of *Quantifying AI Risk*. Hour 1 of 3.

We just spent twenty minutes talking about why this matters. Now we build it. By the end of this notebook you will have a working telemetry emitter, a map from signals to regulatory dimensions, a classification layer that reads the stream against thresholds, a view of which dimensions are covered and which are blind, and a streaming simulation that shows you what drift looks like when it actually arrives.

Everything downstream depends on this. The Bayesian scoring in Hour 2 reads from this stream. The Monte Carlo risk engine in Hour 3 reads from the scores. If the raw signal is not here, nothing after this works. So we start here and we get it right.

## Setup

Standard library only. If you can write a dictionary and a function, you can emit telemetry.

In [ ]:
import random
import json
from datetime import datetime, timezone, timedelta
from collections import defaultdict

# Seed for reproducibility. Remove in production.
random.seed(42)

print("Ready.")

## A single event

Pretend we have a loan approval model running at Acme Corp. Every time it makes a decision, it should emit a structured record. We are going to capture six things per decision:

- what the system decided
- how confident it was
- how long it took
- how far today's inputs are from the training distribution
- whether this decision preserved demographic parity with the rolling distribution
- how much memory the process was using

One note on fairness before we write the code. A single decision cannot be fair or unfair on its own. What we capture per event is a flag. What we report at the dashboard level is a rolling rate over a window. You will see this pattern show up again when we get to classification.

In [ ]:
def emit_event(applicant_id, model_id="loan-approval-v2"):
    """
    One governance telemetry event. In production this lives inside the model
    serving layer and emits to a stream (Kafka, Kinesis, Pub/Sub). Here we
    simulate realistic values for a healthy, in-distribution system.
    """
    return {
        "timestamp":     datetime.now(timezone.utc).isoformat(),
        "model_id":      model_id,
        "applicant_id":  applicant_id,

        # what the system decided
        "decision":      random.choice(["APPROVED", "REJECTED"]),

        # how sure it was. Beta(8,2) means most decisions land in 0.75-0.95.
        "confidence":    round(random.betavariate(8, 2), 3),

        # how long it took. Tight gaussian around 180ms.
        "latency_ms":    round(random.gauss(180, 35), 1),

        # how far today's inputs are from training. Beta(2,40) keeps healthy
        # systems well below 0.10.
        "drift_score":   round(random.betavariate(2, 40), 4),

        # did this decision preserve parity? True ~97% of the time when healthy.
        "fairness_flag": random.random() > 0.03,

        # infrastructure stress. Gaussian around 512MB.
        "memory_mb":     round(random.gauss(512, 30), 1),
    }


event = emit_event("APP-001")
print(json.dumps(event, indent=2))

## What 100 events actually look like

One event is a point. A stream is a shape. Let's look at the shape.

In [ ]:
random.seed(42)
events = [emit_event(f"APP-{i:03d}") for i in range(100)]

confidences  = [e["confidence"]  for e in events]
latencies    = [e["latency_ms"]  for e in events]
drifts       = [e["drift_score"] for e in events]
memories     = [e["memory_mb"]   for e in events]
decisions    = [e["decision"]    for e in events]
fairness_rate = sum(1 for e in events if e["fairness_flag"]) / len(events)


def summary(name, values, healthy_check):
    mean = sum(values) / len(values)
    breached = sum(1 for v in values if healthy_check(v))
    print(f"  {name:14s}  mean={mean:<8.3f}  min={min(values):<8.3f}  max={max(values):<8.3f}  breaches={breached}")


print("100 events, healthy baseline\n")
print("Decisions:")
approvals = sum(1 for d in decisions if d == "APPROVED")
print(f"  approved={approvals}  rejected={100-approvals}\n")

print("Per-event signals:")
summary("confidence",  confidences, lambda v: v < 0.60)
summary("latency_ms",  latencies,   lambda v: v > 300)
summary("drift_score", drifts,      lambda v: v > 0.10)
summary("memory_mb",   memories,    lambda v: v > 600)

print("\nRolling-rate signal:")
print(f"  fairness_rate  {fairness_rate:.1%}  (healthy >= 95%)")

## The signal-to-dimension map

This is the move that makes what we just built actually governance and not just fancy monitoring. Each signal maps to a governance dimension, and each dimension traces back to a requirement in one of the major frameworks.

I am deliberately pointing at the control families (MANAGE, MAP, Annex A) rather than specific sub-control numbers. Sub-controls change between framework versions and depend on your certification scope. Your ISO 42001 lead auditor, or equivalent, does that mapping for your organization. Your job as the engineer is to emit the signal and label the dimension correctly.

In [ ]:
GOVERNANCE_MAP = {
    "confidence": {
        "dimension":   "Reliability & Robustness",
        "anchors":     ["NIST AI RMF MANAGE", "ISO 42001 Annex A"],
        "healthy":     0.70,
        "critical":    0.50,
        "direction":   "higher_is_better",
    },
    "latency_ms": {
        "dimension":   "Operational Health",
        "anchors":     ["NIST AI RMF MANAGE", "ISO 42001 Annex A"],
        "healthy":     300,
        "critical":    500,
        "direction":   "lower_is_better",
    },
    "drift_score": {
        "dimension":   "Data & Model Integrity",
        "anchors":     ["NIST AI RMF MAP", "ISO 42001 Annex A"],
        "healthy":     0.10,
        "critical":    0.20,
        "direction":   "lower_is_better",
    },
    "fairness_flag": {
        "dimension":   "Fairness & Societal Impact",
        "anchors":     ["EU AI Act Article 10", "NIST AI RMF MAP"],
        "healthy":     0.95,   # rolling rate, not per-event
        "critical":    0.85,
        "direction":   "higher_is_better",
    },
    "memory_mb": {
        "dimension":   "Resilience & Continuity",
        "anchors":     ["NIST AI RMF MANAGE", "ISO 42001 Annex A"],
        "healthy":     600,
        "critical":    800,
        "direction":   "lower_is_better",
    },
    "decision": {
        "dimension":   "Transparency & Accountability",
        "anchors":     ["EU AI Act Article 13", "EU AI Act Article 14"],
        "healthy":     None,   # always recorded, no threshold
        "critical":    None,
        "direction":   "record_only",
    },
}


for signal, info in GOVERNANCE_MAP.items():
    print(f"{signal}")
    print(f"  dimension  {info['dimension']}")
    print(f"  anchors    {', '.join(info['anchors'])}")
    if info['direction'] == 'record_only':
        print(f"  threshold  always recorded")
    elif info['direction'] == 'lower_is_better':
        print(f"  healthy    <= {info['healthy']}")
        print(f"  critical   >  {info['critical']}")
    else:
        print(f"  healthy    >= {info['healthy']}")
        print(f"  critical   <  {info['critical']}")
    print()

## Classify every event

Now we read the stream against the map. Every per-event signal gets classified as healthy, warning, or critical. The fairness signal gets classified against its rolling rate, not per-event, because that is what fairness actually is.

This classification step is what feeds the Bayesian scoring engine in Hour 2. One way to think about it: classification turns continuous signals into discrete evidence. Bayesian reasoning updates on discrete evidence. So the shape of what we are building here is already pointing at what comes next.

In [ ]:
def classify(signal_name, value, gov_map):
    info = gov_map[signal_name]
    if info["direction"] == "record_only":
        return "HEALTHY"

    healthy, critical = info["healthy"], info["critical"]

    if info["direction"] == "lower_is_better":
        if value <= healthy:  return "HEALTHY"
        if value <= critical: return "WARNING"
        return "CRITICAL"
    else:  # higher_is_better
        if value >= healthy:  return "HEALTHY"
        if value >= critical: return "WARNING"
        return "CRITICAL"


# Roll per-event signals up by dimension
per_event_signals = ["confidence", "latency_ms", "drift_score", "memory_mb"]
by_dimension = defaultdict(lambda: {"HEALTHY": 0, "WARNING": 0, "CRITICAL": 0})

for event in events:
    for signal in per_event_signals:
        status = classify(signal, event[signal], GOVERNANCE_MAP)
        by_dimension[GOVERNANCE_MAP[signal]["dimension"]][status] += 1

# Transparency is the audit trail itself. Always recorded, always healthy.
by_dimension["Transparency & Accountability"]["HEALTHY"] = len(events)

# Fairness runs against the rolling rate, reported separately.
fairness_status = classify("fairness_flag", fairness_rate, GOVERNANCE_MAP)


print("Dimension status, 100 events\n")

for dim in sorted(by_dimension.keys()):
    counts = by_dimension[dim]
    total = sum(counts.values())
    crit_pct = counts["CRITICAL"] / total * 100
    warn_pct = counts["WARNING"]  / total * 100

    if   crit_pct > 5:   overall = "CRITICAL"
    elif warn_pct > 15:  overall = "WARNING"
    else:                overall = "HEALTHY"

    print(f"  [{overall:8s}]  {dim}")
    print(f"              healthy={counts['HEALTHY']:3d}  warning={counts['WARNING']:3d}  critical={counts['CRITICAL']:3d}")

print(f"\n  [{fairness_status:8s}]  Fairness & Societal Impact")
print(f"              rolling parity rate = {fairness_rate:.1%}  (threshold: healthy >= 95%, critical < 85%)")

## What we are not measuring

Here is the moment most governance assessments turn uncomfortable. Everything we just instrumented covers some dimensions. It does not cover others. The dimensions we are not measuring are blind spots, and a blind spot cannot be scored.

Below is an illustrative list of dimensions a mature framework might cover. Your organization's list may look different. The exercise is the same either way: compare what you have against what you should have, and name the gaps explicitly.

In [ ]:
# An illustrative dimension set. Your framework may define different boundaries.
DIMENSIONS = [
    "Reliability & Robustness",
    "Operational Health",
    "Data & Model Integrity",
    "Fairness & Societal Impact",
    "Resilience & Continuity",
    "Transparency & Accountability",
    "Human Oversight",
    "Content Safety & Truthfulness",
    "Security, Privacy & Data Governance",
    "Regulatory Alignment",
    "Explainability",
]

covered = {info["dimension"] for info in GOVERNANCE_MAP.values()}

covered_count = 0
for dim in DIMENSIONS:
    if dim in covered:
        print(f"  [covered]  {dim}")
        covered_count += 1
    else:
        print(f"  [  GAP  ]  {dim}")

pct = covered_count / len(DIMENSIONS) * 100
print(f"\nCoverage: {covered_count}/{len(DIMENSIONS)} dimensions ({pct:.0f}%)")
print(f"Gaps:     {len(DIMENSIONS) - covered_count} dimensions with no telemetry")
print("\nDimensions without telemetry cannot be scored. They are the blind spots.")

## What drift actually looks like

Static analysis only gets you so far. In production, telemetry flows continuously, and the question is whether your instrumentation will notice when something changes.

We are going to simulate a thousand events. For the first six hundred, the system runs healthy. At event 600, we inject drift. Imagine a new customer segment showing up, or a policy change at the loan origination partner, or just a market shift. The model does not know anything has changed. The telemetry does.

Watch how the signals move together when drift arrives. This is the single most important pattern to carry out of this hour. Governance signals are correlated. When a system starts drifting, it usually also becomes less confident, slower, more resource-stressed, and less fair. A system that monitors each signal independently will throw four separate alerts without ever recognizing the underlying event. A system that scores them jointly catches the pattern. That joint scoring is what we build in Hour 2.

In [ ]:
def emit_event_with_scenario(event_num):
    """
    Events 0-599:    healthy, in-distribution
    Events 600-999:  drift has arrived, every signal degrades together
    """
    event = {
        "timestamp":    (datetime.now(timezone.utc) + timedelta(minutes=event_num)).isoformat(),
        "model_id":     "loan-approval-v2",
        "applicant_id": f"APP-{event_num:04d}",
        "decision":     random.choice(["APPROVED", "REJECTED"]),
    }

    if event_num < 600:
        # Healthy. Matches the baseline we saw earlier.
        event["confidence"]    = round(random.betavariate(8, 2), 3)
        event["latency_ms"]    = round(random.gauss(180, 35), 1)
        event["drift_score"]   = round(random.betavariate(2, 40), 4)
        event["fairness_flag"] = random.random() > 0.03
        event["memory_mb"]     = round(random.gauss(512, 30), 1)
    else:
        # Drift scenario. Every signal degrades.
        event["confidence"]    = round(random.betavariate(3, 3), 3)     # much less sure
        event["latency_ms"]    = round(random.gauss(320, 80), 1)        # slower, noisier
        event["drift_score"]   = round(random.betavariate(8, 10), 4)    # clearly elevated
        event["fairness_flag"] = random.random() > 0.15                 # parity degrading
        event["memory_mb"]     = round(random.gauss(620, 50), 1)        # more pressure
    return event


random.seed(42)
stream = [emit_event_with_scenario(i) for i in range(1000)]


def window_stats(events, label):
    n = len(events)
    print(f"  {label}  (n={n})")
    print(f"    confidence    {sum(e['confidence']  for e in events)/n:.3f}")
    print(f"    drift_score   {sum(e['drift_score'] for e in events)/n:.4f}")
    print(f"    fairness_rate {sum(1 for e in events if e['fairness_flag'])/n:.1%}")
    print(f"    latency_ms    {sum(e['latency_ms']  for e in events)/n:.0f}")
    print(f"    memory_mb     {sum(e['memory_mb']   for e in events)/n:.0f}")
    print()


print("Stream summary: before and after drift injection\n")
window_stats(stream[:600],  "events 0-599    (healthy)")
window_stats(stream[600:],  "events 600-999  (drift)")

print("Every signal moves in the same direction when drift arrives.")
print("Confidence drops. Drift score rises. Fairness degrades. Latency climbs.")
print("Memory pressure grows. These signals are correlated in production.")

## How fast does each signal notice?

One more view before we leave the stream. How many events after drift arrives does each signal's rolling window cross its healthy threshold? This is the detection lag, and minimizing it is the reason Hour 2 exists. A single signal might take dozens of events to trip. Combined probabilistically, the joint evidence crosses faster than any single signal alone. That is the argument for Bayesian scoring, and we will prove it tomorrow.

In [ ]:
DRIFT_START = 600
window = 50
lags = {}

for signal, info in GOVERNANCE_MAP.items():
    if info["direction"] == "record_only":
        continue

    for i in range(max(DRIFT_START, window), len(stream)):
        w = stream[i-window:i]
        if signal == "fairness_flag":
            value = sum(1 for e in w if e["fairness_flag"]) / window
        else:
            value = sum(e[signal] for e in w) / window

        if info["direction"] == "lower_is_better":
            breached = value > info["healthy"]
        else:
            breached = value < info["healthy"]

        if breached:
            lags[signal] = i - DRIFT_START
            break


print(f"Drift injected at event {DRIFT_START}. Rolling window = {window} events.\n")
print("Events after drift before each signal breaches its healthy threshold:\n")

for signal in sorted(lags, key=lags.get):
    print(f"  {signal:14s}  +{lags[signal]:3d} events")

print("\nDirect signals (drift, fairness) trip fastest. Downstream consequences")
print("(confidence, memory, latency) take longer. Hour 2 is about making the")
print("joint evidence cross threshold faster than any single signal can alone.")

## Your turn: close a coverage gap

The coverage analysis showed five dimensions with zero telemetry. Pick one and close it.

The five options:
- **Human Oversight** — does the system escalate to a person when appropriate?
- **Content Safety & Truthfulness** — does the system produce harmful or false output?
- **Security, Privacy & Data Governance** — is sensitive data being handled correctly?
- **Regulatory Alignment** — can each decision be traced back to a specific regulatory requirement?
- **Explainability** — can the system produce an explanation for its decision?

What you do:

1. Design the signal. Name it. Pick a type. Decide what healthy looks like and what critical looks like.
2. Implement it. Copy `emit_event()` into `emit_event_v2()` and add your signal.
3. Map it. Add an entry to `GOVERNANCE_MAP_V2` with a defensible regulatory anchor.
4. Classify it. Run the pipeline on 100 fresh events.
5. Justify the thresholds in writing. Three to five sentences in the cell below the code. Name the specific risk each threshold guards against. Cite a regulatory anchor. Describe what would have to change in the world for you to revise it.

No solution code in this notebook. The reason is simple. In production, every new governance signal requires engineering judgment about where to set the thresholds and how to defend them. A regulator will ask. A plaintiff's attorney will ask. The written justification is the artifact that survives both conversations. Practice writing that justification now, while the stakes are low.

In [ ]:
# Your implementation

def emit_event_v2(applicant_id, model_id="loan-approval-v2"):
    # Copy the body of emit_event()
    # Add your new signal
    pass


GOVERNANCE_MAP_V2 = {
    # Copy GOVERNANCE_MAP entries
    # Add your new entry with dimension, anchors, thresholds, and direction
}


# Generate 100 fresh events, classify against GOVERNANCE_MAP_V2,
# and print coverage. Should be 7 of 11 dimensions instead of 6 of 11.


### Your threshold justification

*Write three to five sentences here. The specific risk each threshold guards against. At least one regulatory anchor. What would have to change for you to revise the thresholds. This is the written rationale that goes into an ISO 42001 technical documentation file. Practice the habit now.*

## What we just built

A telemetry layer. Specifically:

- An emitter that captures six signals on every decision
- A map that connects each signal to a governance dimension and a framework anchor
- A classifier that reads the stream against thresholds, handling per-event and rolling-rate signals differently
- A coverage view that names the blind spots
- A streaming simulation that shows what drift looks like and how fast each signal notices

This is the raw data layer. Everything in Hour 2 and Hour 3 reads from this stream. The Bayesian scoring turns the stream into a continuously updated trust score. The Monte Carlo simulation turns the trust score into a dollar-denominated risk number. But the stream is the foundation. If the signal is not captured at the moment the decision is made, nothing downstream can recover it.

Remember the two cases we opened with. Air Canada and Workday both lost on the same missing evidence. What we just built is the smallest working version of what was missing. Take it back on Monday. Extend it. Make it real.

On to Hour 2.